# sanoTTS — Indonesian audio enhancer training (point 3)

Trains the tiny residual waveform enhancer (`tools/train_roota_audio_enhancer.py`, <100k params) for the `id` voice.

**Data:** ~160 Indonesian sentences rendered twice — clean by the Piper teacher (`id_ID-news_tts-medium`), noisy by our 1.46M student (numpy engine, espeak ids — same ids the released voice was trained on).

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# @title 1. Clone repo + install deps (~2 min)
REPO_URL = "https://github.com/wafik/sanoTTS.git"  # @param {type:"string"}
BRANCH = "master"  # @param {type:"string"}

!git clone -q --depth 1 -b $BRANCH $REPO_URL /content/sanoTTS
%cd /content/sanoTTS

# trainer deps: torch is preinstalled on Colab; piper for teacher render; soundfile for wav io
!pip install -q piper-tts onnxruntime soundfile
# our numpy engine + espeak backend for student render
!pip install -q phonemizer-fork espeakng-loader
!apt-get -qq install -y espeak-ng > /dev/null
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# @title 2. Download the Indonesian Piper teacher (~60 MB)
import urllib.request, pathlib
from pathlib import Path

TEACHER_DIR = Path("models/teachers/id_ID-news_tts-medium")
TEACHER_DIR.mkdir(parents=True, exist_ok=True)
BASE = "https://huggingface.co/rhasspy/piper-voices/resolve/main/id/id_ID/news_tts/medium/"
for fn in ["id_ID-news_tts-medium.onnx", "id_ID-news_tts-medium.onnx.json"]:
    dst = TEACHER_DIR / fn
    if not dst.exists():
        urllib.request.urlretrieve(BASE + fn + "?download=true", dst)
    print(dst, dst.stat().st_size, "bytes")

In [ ]:
# @title 3. Indonesian text corpus (~160 varied sentences)
import json
from pathlib import Path

openers = ["Halo!", "Selamat pagi!", "Selamat siang!", "Selamat sore!", "Selamat malam!",
           "Terima kasih!", "Permisi,", "Hati-hati!", "Semoga harimu menyenangkan."]
subjects = ["Kami", "Mereka", "Anak-anak", "Para siswa", "Keluarga ini", "Petani itu",
            "Nelayan di desa", "Guru kami", "Dokter muda itu", "Warga kampung"]
verbs = ["pergi ke pasar", "menanam padi di sawah", "membaca buku di perpustakaan",
         "memasak sayur di dapur", "bermain bola di lapangan", "menyusun rencana kerja",
         "mengunjungi museum sejarah", "menyeberangi jembatan tua", "membersihkan halaman rumah",
         "belajar bahasa asing", "menulis surat untuk sahabat", "menonton film bersama"]
closers = ["setiap pagi.", "kemarin sore.", "dengan gembira.", "sangat rajin.",
           "bersama teman-temannya.", "sebelum matahari terbit.", "di akhir pekan.",
           "tanpa terburu-buru.", "sambil bercanda.", "hampir setiap hari."]
facts = [
 "Indonesia memiliki lebih dari tujuh belas ribu pulau.",
 "Jakarta adalah pusat pemerintahan negara.",
 "Hujan turun hampir setiap sore di bulan Januari.",
 "Pasar tradisional buka sejak pagi buta.",
 "Gunung tertinggi di Pulau Jawa berdiri megah di timur.",
 "Anak-anak berlari mengejar layangan di padang terbuka.",
 "Kopi susu gula aren menjadi minuman favorit banyak orang.",
 "Kereta cepat menghubungkan dua kota besar dengan cepat.",
 "Bahasa daerah tetap hidup di lingkungan keluarga.",
 "Musim panen membawa kegembiraan bagi warga desa.",
 "Teknologi membuat komunikasi menjadi lebih mudah.",
 "Buku adalah jendela menuju dunia yang luas.",
]
questions = [
 "Apakah kamu sudah makan hari ini?",
 "Kapan kereta berikutnya datang?",
 "Di mana alamat kantor pos yang terdekat?",
 "Berapa harga tiket masuk museum ini?",
 "Mengapa langit menjadi merah saat senja?",
]

texts = []
for o in openers:
    texts.append(f"{o} {facts[len(texts) % len(facts)]}")
for s in subjects:
    for v in verbs[:6]:
        texts.append(f"{s} {v} {closers[len(texts) % len(closers)]}")
texts.extend(facts); texts.extend(questions)
extras = [
 "Nomor telepon darurat adalah satu satu dua.",
 "Hari ini tanggal dua puluh delapan Agustus dua ribu dua puluh enam.",
 "Dia membaca nomor halaman tiga puluh empat dengan lantang.",
 "Kode pos kantor kelurahan itu delapan puluh satu dua puluh dua.",
]
texts.extend(extras)
# dedupe + cap
seen, corpus = set(), []
for t in texts:
    if t not in seen:
        seen.add(t); corpus.append(t)
corpus = corpus[:160]
Path("corpus").mkdir(exist_ok=True)
with open("corpus/id_sentences.jsonl", "w", encoding="utf-8") as f:
    for i, t in enumerate(corpus):
        f.write(json.dumps({"row_id": f"id-{i:04d}", "text": t}, ensure_ascii=False) + "\n")
print(len(corpus), "sentences written")
for t in corpus[:3]: print(" ", t)

In [ ]:
# @title 4. Render teacher (clean) + student (noisy) pairs → manifests (~8-15 min on T4)
import json, sys, wave
from pathlib import Path
import numpy as np
from piper import PiperVoice

sys.path.insert(0, "pypkg")
from sanotts.engine import Synthesizer
from sanotts.cli import write_wav

SR = 22050
AUDIO = Path("artifacts/enhancer-id/audio"); AUDIO.mkdir(parents=True, exist_ok=True)

# clean lane: piper teacher, deterministic (native rate 22050, verified from the onnx.json).
# piper-tts 1.7 (piper1-gpl) API: synthesize(text) yields AudioChunk with .audio_float_array.
voice = PiperVoice.load("models/teachers/id_ID-news_tts-medium/id_ID-news_tts-medium.onnx")
# noisy lane: our student (downloads id voice into ~/.cache/sanotts on first call)
student = Synthesizer("id")

def render_teacher(text: str, path: Path) -> None:
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
        for chunk in voice.synthesize(text):
            pcm = (np.clip(chunk.audio_float_array, -1.0, 1.0) * 32767.0).astype("<i2")
            w.writeframes(pcm.tobytes())

rows = [json.loads(l) for l in open("corpus/id_sentences.jsonl", encoding="utf-8")]
manifest = []
for i, row in enumerate(rows):
    text = row["text"]
    t_wav = AUDIO / f"{row['row_id']}-teacher.wav"
    s_wav = AUDIO / f"{row['row_id']}-student.wav"
    try:
        render_teacher(text, t_wav)
        r = student.synthesize(text)
        write_wav(s_wav, r.audio, r.sample_rate)
    except Exception as e:
        print("skip", row["row_id"], e); continue
    manifest.append({"row_id": row["row_id"], "text": text,
                     "teacher_audio": str(t_wav), "noisy_audio": str(s_wav)})
    if (i+1) % 20 == 0: print(i+1, "/", len(rows))

# 85/15 split
n_eval = max(8, len(manifest) // 7)
eval_rows, train_rows = manifest[:n_eval], manifest[n_eval:]
Path("artifacts/enhancer-id").mkdir(parents=True, exist_ok=True)
(Path("artifacts/enhancer-id/train.jsonl")).write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in train_rows) + "\n", encoding="utf-8")
(Path("artifacts/enhancer-id/eval.jsonl")).write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in eval_rows) + "\n", encoding="utf-8")
print(f"train={len(train_rows)} eval={len(eval_rows)}")

In [ ]:
# @title 5. Train the enhancer (~5-10 min on T4; <100k params)
!python tools/train_roota_audio_enhancer.py \
  --train-manifest artifacts/enhancer-id/train.jsonl \
  --eval-manifest artifacts/enhancer-id/eval.jsonl \
  --out-dir artifacts/enhancer-id/run \
  --steps 1200 --batch-size 8 --log-every 100

In [ ]:
# @title 6. A/B listen: raw vs enhanced on eval rows
import sys, torch, numpy as np, soundfile as sf
sys.path.insert(0, ".")
from tools.train_roota_audio_enhancer import TinyResidualEnhancer, load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, _ = load_checkpoint("artifacts/enhancer-id/run/audio-enhancer.pt", device)
import json
rows = [json.loads(l) for l in open("artifacts/enhancer-id/eval.jsonl", encoding="utf-8")][:5]
OUT = Path("artifacts/enhancer-id/ab"); OUT.mkdir(parents=True, exist_ok=True)
for r in rows:
    noisy, sr = sf.read(r["noisy_audio"], dtype="float32")
    x = torch.as_tensor(noisy, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)
    with torch.no_grad():
        enhanced, _ = model(x)
    y = enhanced.squeeze().cpu().numpy()
    sf.write(OUT / (Path(r["row_id"]).name + "-raw.wav"), noisy, sr)
    sf.write(OUT / (Path(r["row_id"]).name + "-enhanced.wav"), y, sr)
print("wrote A/B wavs to", OUT)
import shutil
shutil.make_archive("/content/enhancer-id", "zip", "artifacts/enhancer-id/run")
shutil.make_archive("/content/enhancer-id-ab", "zip", "artifacts/enhancer-id/ab")
print("download: /content/enhancer-id.zip (checkpoint) + /content/enhancer-id-ab.zip (A/B wavs)")

## Next (local)
1. Download both zips. `audio-enhancer.pt` goes to `artifacts/enhancer-id/run/` locally.
2. Listen to the A/B wavs. Only ship the enhancer if `enhanced` clearly beats `raw` on your ears.
3. Wiring the checkpoint into the numpy/wasm runtime is a separate step (the trainer saves a torch state_dict; runtime integration needs a port script — say the word and we'll build it).